In [1]:
install.packages("languageserver")

Installing package into ‘/home/nk1125/miniconda3/envs/clean_r_env/lib/R/library’
(as ‘lib’ is unspecified)

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [2]:
library(languageserver)

In [1]:
library(dplyr)
library(readr)
 
# ---- 1. Read both tables --------------------------------------------------
# EFP metadata + monthly meteo (keys: SITE_ID, YEAR, uppercase)
efp <- read_csv("fluxnet_2017_2025_V02/EFP_outputs_corrected/EFP_metadata_monthlyMeteo_WIDE.csv")
 
# Mortality / disturbance table (keys: site_id, year, lowercase)
mort <- read_csv("derived_tables/final_disturbance_v2_multibuffer_with_mortality_intensity.csv")
 
# ---- 2. Harmonise the key columns ----------------------------------------
# Rename mortality keys to match the EFP table so we can join cleanly.
mort <- mort %>%
  rename(SITE_ID = site_id,
         YEAR    = year)
 
# ---- 3. Join --------------------------------------------------------------
# left_join keeps every EFP row and attaches mortality columns where the
# (SITE_ID, YEAR) pair exists in the mortality table.
#   - 184 EFP sites, all present in mortality
#   - 1179 of 1180 EFP site-years match; AU-Cpr 2025 has no mortality row
#     (mortality data for that site ends in 2024) -> mortality cols = NA there
combined <- efp %>%
  left_join(mort, by = c("SITE_ID", "YEAR"))
 
# ---- 4. Quick sanity check ------------------------------------------------
message("EFP rows:        ", nrow(efp))
message("Combined rows:   ", nrow(combined))
# Use forest_mean_pct_500m (present in every real mortality row) to detect
# true join failures. Do NOT use mortality_intensity_*, which is legitimately
# NA at zero-forest-cover sites and first-year rows even when the join succeeds.
message("True join failures (expected 1, AU-Cpr 2025): ",
        sum(is.na(combined$forest_mean_pct_500m)))
 
# ---- 5. Write out ---------------------------------------------------------
write_csv(combined, "derived_tables/outputs_afterEGU_results/EFP_mortality_combined.csv")
message("Wrote EFP_mortality_combined.csv with ", ncol(combined), " columns")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Rows: 1180 Columns: 156
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (6): SITE_ID, precipAvail, Gavail, CO2avail, status, IGBP
dbl (150): YEAR, uWUE, WUE, ETmax, GSmax, G1, EF, EFampl, GPPsat, NEPmax, Rb...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 3330 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): site_id
dbl (51): year, forest_mean_pct_100m, deadwood_mean_pct_100m, mortality_inte...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
EFP rows

In [4]:
length(unique(combined$SITE_ID))

[1] 184